In [1]:
import pandas as pd
import numpy as np

associated_infections = pd.read_csv(
    'Healthcare_Associated_Infections-Hospital.csv',
    dtype={'Facility ID': str},
    low_memory=False
)

print(f"Raw shape: {associated_infections.shape}")
associated_infections.head()

Raw shape: (172404, 15)


,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Measure ID,Measure Name,Compared to National,Score,Footnote,Start Date,End Date
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,HAI_1_CILOWER,Central Line Associated Bloodstream Infection ...,No Different than National Benchmark,0.133,NaN,04/01/2024,03/31/2025
1,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,HAI_1_CIUPPER,Central Line Associated Bloodstream Infection ...,No Different than National Benchmark,1.008,NaN,04/01/2024,03/31/2025
2,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,HAI_1_DOPC,Central Line Associated Bloodstream Infection:...,No Different than National Benchmark,9078,NaN,04/01/2024,03/31/2025
3,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,HAI_1_ELIGCASES,Central Line Associated Bloodstream Infection ...,No Different than National Benchmark,9.568,NaN,04/01/2024,03/31/2025
4,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,HAI_1_NUMERATOR,Central Line Associated Bloodstream Infection ...,No Different than National Benchmark,4,NaN,04/01/2024,03/31/2025


In [2]:
print("--- Raw nulls ---")
print(associated_infections.isnull().sum())

print("\n--- Raw dtypes ---")
print(associated_infections.dtypes)

--- Raw nulls ---
Facility ID                 0
Facility Name               0
Address                     0
City/Town                   0
State                       0
ZIP Code                    0
County/Parish               0
Telephone Number            0
Measure ID                  0
Measure Name                0
Compared to National        0
Score                    2118
Footnote                92515
Start Date                  0
End Date                    0
dtype: int64

--- Raw dtypes ---
Facility ID             object
Facility Name           object
Address                 object
City/Town               object
State                   object
ZIP Code                 int64
County/Parish           object
Telephone Number        object
Measure ID              object
Measure Name            object
Compared to National    object
Score                   object
Footnote                object
Start Date              object
End Date                object
dtype: object


In [3]:
associated_infections.columns = (
    associated_infections.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('/', '_')
    .str.replace(r'[^\w]', '_', regex=True)
)

print(associated_infections.columns.tolist())

['facility_id', 'facility_name', 'address', 'city_town', 'state', 'zip_code', 'county_parish', 'telephone_number', 'measure_id', 'measure_name', 'compared_to_national', 'score', 'footnote', 'start_date', 'end_date']


In [4]:
CMS_NULLS = {
    'Not Available': np.nan,
    'Not Applicable': np.nan,
    'Too Few to Report': np.nan,
    'N/A': np.nan
}

associated_infections = associated_infections.replace(CMS_NULLS)

print("CMS placeholders replaced ")
print(f"Nulls after replace:\n{associated_infections.isnull().sum()}")

CMS placeholders replaced 
Nulls after replace:
facility_id                  0
facility_name                0
address                      0
city_town                    0
state                        0
zip_code                     0
county_parish                0
telephone_number             0
measure_id                   0
measure_name                 0
compared_to_national    104772
score                    76048
footnote                 92515
start_date                   0
end_date                     0
dtype: int64


In [5]:
associated_infections['facility_id'] = (
    associated_infections['facility_id']
    .str.strip()
    .str.zfill(6)
)

print("facility_id sample:", associated_infections['facility_id'].head().tolist())

facility_id sample: ['010001', '010001', '010001', '010001', '010001']


In [6]:
associated_infections['score'] = pd.to_numeric(associated_infections['score'], errors='coerce')

print("Numeric dtypes after cast:")
print(associated_infections['score'].dtypes)

Numeric dtypes after cast:
float64


In [7]:
associated_infections['is_suppressed'] = associated_infections['footnote'].notna()

print("Suppression breakdown:")
print(associated_infections['is_suppressed'].value_counts())

Suppression breakdown:
is_suppressed
False    92515
True     79889
Name: count, dtype: int64


In [8]:
associated_infections['compared_to_national'] = (
    associated_infections['compared_to_national']
    .str.strip()
)

print("compared_to_national values:")
print(associated_infections['compared_to_national'].value_counts(dropna=False))

compared_to_national values:
compared_to_national
NaN                                     104772
No Different than National Benchmark     46758
Better than the National Benchmark       19608
Worse than the National Benchmark         1266
Name: count, dtype: int64


In [9]:
cols_to_drop = [
    'facility_name', 'address', 'city_town',
    'state', 'zip_code', 'county_parish', 'telephone_number',
    'start_date', 'end_date',  
    'footnote'                 
]

associated_infections = associated_infections.drop(columns=cols_to_drop, errors='ignore')

print(f"Shape after dropping: {associated_infections.shape}")
print(associated_infections.columns.tolist())

Shape after dropping: (172404, 6)
['facility_id', 'measure_id', 'measure_name', 'compared_to_national', 'score', 'is_suppressed']


In [10]:
dupes = associated_infections.duplicated(subset=['facility_id', 'measure_id']).sum()
print(f"True duplicates on composite key: {dupes}")

True duplicates on composite key: 0


In [11]:
print(f"Final shape      : {associated_infections.shape}")
print(f"Unique hospitals : {associated_infections['facility_id'].nunique()}")
print(f"Unique measures  : {associated_infections['measure_id'].nunique()}")

print("\n--- Nulls ---")
print(associated_infections.isnull().sum())

print("\n--- Measures available ---")
print(associated_infections['measure_id'].value_counts())

Final shape      : (172404, 6)
Unique hospitals : 4789
Unique measures  : 36

--- Nulls ---
facility_id                  0
measure_id                   0
measure_name                 0
compared_to_national    104772
score                    76048
is_suppressed                0
dtype: int64

--- Measures available ---
measure_id
HAI_1_CILOWER      4789
HAI_1_CIUPPER      4789
HAI_4_DOPC         4789
HAI_4_ELIGCASES    4789
HAI_4_NUMERATOR    4789
HAI_4_SIR          4789
HAI_5_CILOWER      4789
HAI_5_CIUPPER      4789
HAI_5_DOPC         4789
HAI_5_ELIGCASES    4789
HAI_5_NUMERATOR    4789
HAI_5_SIR          4789
HAI_6_CILOWER      4789
HAI_6_CIUPPER      4789
HAI_6_DOPC         4789
HAI_6_ELIGCASES    4789
HAI_6_NUMERATOR    4789
HAI_4_CIUPPER      4789
HAI_4_CILOWER      4789
HAI_3_SIR          4789
HAI_2_DOPC         4789
HAI_1_DOPC         4789
HAI_1_ELIGCASES    4789
HAI_1_NUMERATOR    4789
HAI_1_SIR          4789
HAI_2_CILOWER      4789
HAI_2_CIUPPER      4789
HAI_2_ELIGCASES    478

In [12]:
comp_scores = associated_infections.pivot_table(
    index='facility_id',
    columns='measure_id',
    values='score',
    aggfunc='first'
).reset_index()

comp_scores.columns = (
    ['facility_id'] +
    ['infection_score_' + col.lower() for col in comp_scores.columns[1:]]
)

comp_flags = associated_infections.pivot_table(
    index='facility_id',
    columns='measure_id',
    values='compared_to_national',
    aggfunc='first'
).reset_index()

comp_flags.columns = (
    ['facility_id'] +
    ['infection_flag_' + col.lower() for col in comp_flags.columns[1:]]
)

infections_wide = comp_scores.merge(comp_flags, on='facility_id', how='left')

print(f"Wide table shape: {infections_wide.shape}")
infections_wide.head()

Wide table shape: (4288, 73)


,facility_id,infection_score_hai_1_cilower,infection_score_hai_1_ciupper,infection_score_hai_1_dopc,infection_score_hai_1_eligcases,infection_score_hai_1_numerator,infection_score_hai_1_sir,infection_score_hai_2_cilower,infection_score_hai_2_ciupper,infection_score_hai_2_dopc,...,infection_flag_hai_5_dopc,infection_flag_hai_5_eligcases,infection_flag_hai_5_numerator,infection_flag_hai_5_sir,infection_flag_hai_6_cilower,infection_flag_hai_6_ciupper,infection_flag_hai_6_dopc,infection_flag_hai_6_eligcases,infection_flag_hai_6_numerator,infection_flag_hai_6_sir
0,010001,0.133,1.008,9078.0,9.568,4.0,0.418,0.032,0.341,16727.0,...,Better than the National Benchmark,Better than the National Benchmark,Better than the National Benchmark,Better than the National Benchmark,Better than the National Benchmark,Better than the National Benchmark,Better than the National Benchmark,Better than the National Benchmark,Better than the National Benchmark,Better than the National Benchmark
1,010005,0.454,3.450,4592.0,2.797,4.0,1.430,0.268,2.870,6224.0,...,No Different than National Benchmark,No Different than National Benchmark,No Different than National Benchmark,No Different than National Benchmark,No Different than National Benchmark,No Different than National Benchmark,No Different than National Benchmark,No Different than National Benchmark,No Different than National Benchmark,No Different than National Benchmark
2,010006,NaN,0.697,3887.0,4.301,0.0,0.000,0.005,0.518,7376.0,...,Better than the National Benchmark,Better than the National Benchmark,Better than the National Benchmark,Better than the National Benchmark,Better than the National Benchmark,Better than the National Benchmark,Better than the National Benchmark,Better than the National Benchmark,Better than the National Benchmark,Better than the National Benchmark
3,010007,NaN,NaN,252.0,0.156,0.0,NaN,NaN,NaN,1430.0,...,NaN,NaN,NaN,NaN,No Different than National Benchmark,No Different than National Benchmark,No Different than National Benchmark,No Different than National Benchmark,No Different than National Benchmark,No Different than National Benchmark
4,010008,NaN,NaN,30.0,0.017,0.0,NaN,NaN,NaN,326.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
infections_wide.to_csv('associated_infections_cleaned.csv', index=False)
print("Saved as associated_infections_cleaned.csv")

Saved as associated_infections_cleaned.csv


In [14]:
print(infections_wide.columns.tolist())

['facility_id', 'infection_score_hai_1_cilower', 'infection_score_hai_1_ciupper', 'infection_score_hai_1_dopc', 'infection_score_hai_1_eligcases', 'infection_score_hai_1_numerator', 'infection_score_hai_1_sir', 'infection_score_hai_2_cilower', 'infection_score_hai_2_ciupper', 'infection_score_hai_2_dopc', 'infection_score_hai_2_eligcases', 'infection_score_hai_2_numerator', 'infection_score_hai_2_sir', 'infection_score_hai_3_cilower', 'infection_score_hai_3_ciupper', 'infection_score_hai_3_dopc', 'infection_score_hai_3_eligcases', 'infection_score_hai_3_numerator', 'infection_score_hai_3_sir', 'infection_score_hai_4_cilower', 'infection_score_hai_4_ciupper', 'infection_score_hai_4_dopc', 'infection_score_hai_4_eligcases', 'infection_score_hai_4_numerator', 'infection_score_hai_4_sir', 'infection_score_hai_5_cilower', 'infection_score_hai_5_ciupper', 'infection_score_hai_5_dopc', 'infection_score_hai_5_eligcases', 'infection_score_hai_5_numerator', 'infection_score_hai_5_sir', 'infectio